In [9]:
# @title
import sqlite3
import pandas as pd
from google.colab import drive

# --- 1. Параметры ---
drive.mount('/content/drive')
DB_PATH = '/content/drive/MyDrive/Data_Bases/rpl.db'

# Helper to get display width of string, accounting for emojis (assuming 2-cell width)
def get_display_width(text):
    width = 0
    for char in str(text):
        # Heuristic for common emojis to count as 2 display cells
        if char in ['⚽', '👟', '🟨', '🟥']:
            width += 2
        else:
            width += 1
    return width

try:
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    print(f"✅ Успешно подключено к {DB_PATH}")
except Exception as e:
    print(f"❌ Ошибка подключения к базе данных: {e}")
    # Если не удалось подключиться, дальнейший код не имеет смысла
    raise SystemExit("Работа скрипта прервана.")

print("Successfully connected to FA_Premier_League.db and created a cursor object.")

# Get team IDs from user input
team_id_1 = int(input("Введите ID первой команды для сравнения: "))
team_id_2 = int(input("Введите ID второй команды для сравнения: "))

print(f"Вы выбрали команды с ID: {team_id_1} и {team_id_2}")

# Retrieve team names
cursor.execute("SELECT team_name FROM teams WHERE team_id = ?", (team_id_1,))
team_name_1 = cursor.fetchone()[0]

cursor.execute("SELECT team_name FROM teams WHERE team_id = ?", (team_id_2,))
team_name_2 = cursor.fetchone()[0]

print(f"Команда 1: {team_name_1} (ID: {team_id_1})")
print(f"Команда 2: {team_name_2} (ID: {team_id_2})")

# Get league standings
query_standings = '''
WITH MatchScores AS (
    SELECT
        M.match_id,
        M.team1_id AS home_team_id,
        MS_home.goals AS home_goals,
        M.team2_id AS away_team_id,
        MS_away.goals AS away_goals
    FROM Matches AS M
    JOIN MatchStats AS MS_home ON M.match_id = MS_home.match_id AND M.team1_id = MS_home.team_id
    JOIN MatchStats AS MS_away ON M.match_id = MS_away.match_id AND M.team2_id = MS_away.team_id
    WHERE season = (SELECT max(season) FROM Matches)
),
AllTeamMatchStats AS (
    -- Home games stats for all teams
    SELECT
        home_team_id AS team_id,
        1 AS played,
        CASE WHEN home_goals > away_goals THEN 1 ELSE 0 END AS won,
        CASE WHEN home_goals = away_goals THEN 1 ELSE 0 END AS drawn,
        CASE WHEN home_goals < away_goals THEN 1 ELSE 0 END AS lost,
        home_goals AS goals_for,
        away_goals AS goals_against,
        (home_goals - away_goals) AS goal_difference,
        CASE WHEN home_goals > away_goals THEN 3 WHEN home_goals = away_goals THEN 1 ELSE 0 END AS points
    FROM MatchScores

    UNION ALL

    -- Away games stats for all teams
    SELECT
        away_team_id AS team_id,
        1 AS played,
        CASE WHEN away_goals > home_goals THEN 1 ELSE 0 END AS won,
        CASE WHEN away_goals = home_goals THEN 1 ELSE 0 END AS drawn,
        CASE WHEN away_goals < home_goals THEN 1 ELSE 0 END AS lost,
        away_goals AS goals_for,
        home_goals AS goals_against,
        (away_goals - home_goals) AS goal_difference,
        CASE WHEN away_goals > home_goals THEN 3 WHEN away_goals = home_goals THEN 1 ELSE 0 END AS points
    FROM MatchScores
),
FullLeagueStandings AS (
    SELECT
        T.team_id,
        T.team_name,
        SUM(tms.points) AS points,
        SUM(tms.played) AS played,
        SUM(tms.won) AS won,
        SUM(tms.drawn) AS drawn,
        SUM(tms.lost) AS lost,
        SUM(tms.goals_for) AS goals_for,
        SUM(tms.goals_against) AS goals_against,
        SUM(tms.goal_difference) AS goal_difference,
        RANK() OVER (ORDER BY SUM(tms.points) DESC, SUM(tms.goal_difference) DESC, SUM(tms.goals_for) DESC) AS position
    FROM AllTeamMatchStats AS tms
    JOIN Teams AS T ON tms.team_id = T.team_id
    GROUP BY T.team_id, T.team_name
)
SELECT
    position,
    team_name,
    played,
    won,
    drawn,
    lost,
    points,
    goals_for || '-' || goals_against AS goals
FROM FullLeagueStandings
WHERE team_id IN (?, ?)
ORDER BY position ASC;
'''

cursor.execute(query_standings, (team_id_1, team_id_2))
standings_data = cursor.fetchall()

standings_df = pd.DataFrame(standings_data, columns=['position', 'team_name', 'played', 'won', 'drawn', 'lost', 'points', 'goals'])

print("\nТурнирная таблица:")
print(standings_df.to_string(index=False))

# Get last five matches
query_recent_matches = '''
WITH MatchDetails AS (
    SELECT
        M.match_id,
        M.match_date,
        M.team1_id AS home_team_id,
        T_home.team_name AS home_team_name,
        MS_home.goals AS home_goals,
        M.team2_id AS away_team_id,
        T_away.team_name AS away_team_name,
        MS_away.goals AS away_goals
    FROM Matches AS M
    JOIN Teams AS T_home ON M.team1_id = T_home.team_id
    JOIN Teams AS T_away ON M.team2_id = T_away.team_id
    JOIN MatchStats AS MS_home ON M.match_id = MS_home.match_id AND M.team1_id = MS_home.team_id
    JOIN MatchStats AS MS_away ON M.match_id = MS_away.match_id AND M.team2_id = MS_away.team_id
)
SELECT
    match_date,
    home_team_name,
    home_goals,
    away_team_name,
    away_goals,
    CASE
        WHEN home_team_id = ? AND home_goals > away_goals THEN 'Победа'
        WHEN home_team_id = ? AND home_goals < away_goals THEN 'Поражение'
        WHEN away_team_id = ? AND away_goals > home_goals THEN 'Победа'
        WHEN away_team_id = ? AND away_goals < home_goals THEN 'Поражение'
        ELSE 'Ничья'
    END AS result_for_team
FROM MatchDetails
WHERE home_team_id = ? OR away_team_id = ?
ORDER BY match_date DESC
LIMIT 5
'''

# Get last 5 matches for team_id_1
cursor.execute(query_recent_matches, (team_id_1, team_id_1, team_id_1, team_id_1, team_id_1, team_id_1))
recent_matches_data_1 = cursor.fetchall()

recent_matches_df_1 = pd.DataFrame(recent_matches_data_1, columns=['Дата', 'Домашняя команда', 'Голы домашней команды', 'Гостевая команда', 'Голы гостевой команды', 'Результат для команды'])

# Get last 5 matches for team_id_2
cursor.execute(query_recent_matches, (team_id_2, team_id_2, team_id_2, team_id_2, team_id_2, team_id_2))
recent_matches_data_2 = cursor.fetchall()

recent_matches_df_2 = pd.DataFrame(recent_matches_data_2, columns=['Дата', 'Домашняя команда', 'Голы домашней команды', 'Гостевая команда', 'Голы гостевой команды', 'Результат для команды'])

print(f"\nПоследние 5 матчей для {team_name_1}:")
print(recent_matches_df_1.to_string(index=False))

print(f"\nПоследние 5 матчей для {team_name_2}:")
print(recent_matches_df_2.to_string(index=False))

# Get top scorers and assistants
query_top_scorers = '''
SELECT
    P.player_name,
    SUM(PS.goals) AS goals
FROM PlayerStats AS PS JOIN Matches ON Matches.match_id = PS.match_id
JOIN Players AS P ON PS.player_id = P.player_id
WHERE PS.team_id = ? AND season = (SELECT max(season) FROM Matches)
GROUP BY P.player_name
HAVING SUM(PS.goals) > 0
ORDER BY SUM(PS.goals) DESC
LIMIT 3;
'''

query_top_assists = '''
SELECT
    P.player_name,
    SUM(PS.assists) AS assists
FROM PlayerStats AS PS JOIN Matches ON Matches.match_id = PS.match_id
JOIN Players AS P ON PS.player_id = P.player_id
WHERE PS.team_id = ? AND season = (SELECT max(season) FROM Matches)
GROUP BY P.player_name
HAVING SUM(PS.assists) > 0
ORDER BY SUM(PS.assists) DESC
LIMIT 3;
'''

query_overall_top_scorers = '''
SELECT
    P.player_name,
    SUM(PS.goals) AS goals,
    RANK() OVER (ORDER BY SUM(PS.goals) DESC) AS overall_rank
FROM PlayerStats AS PS JOIN Matches ON Matches.match_id = PS.match_id
JOIN Players AS P ON PS.player_id = P.player_id
WHERE season = (SELECT max(season) FROM Matches)
GROUP BY P.player_name
HAVING SUM(PS.goals) > 0
ORDER BY SUM(PS.goals) DESC;
'''

query_overall_top_assists = '''
SELECT
    P.player_name,
    SUM(PS.assists) AS assists,
    RANK() OVER (ORDER BY SUM(PS.assists) DESC) AS overall_rank
FROM PlayerStats AS PS JOIN Matches ON Matches.match_id = PS.match_id
JOIN Players AS P ON PS.player_id = P.player_id
WHERE season = (SELECT max(season) FROM Matches)
GROUP BY P.player_name
HAVING SUM(PS.assists) > 0
ORDER BY SUM(PS.assists) DESC;
'''

# Get top scorers for team_id_1
cursor.execute(query_top_scorers, (team_id_1,))
top_scorers_1 = cursor.fetchall()
top_scorers_df_1 = pd.DataFrame(top_scorers_1, columns=['Игрок', 'Голы'])

# Get top assists for team_id_1
cursor.execute(query_top_assists, (team_id_1,))
top_assists_1 = cursor.fetchall()
top_assists_df_1 = pd.DataFrame(top_assists_1, columns=['Игрок', 'Ассисты'])

# Get top scorers for team_id_2
cursor.execute(query_top_scorers, (team_id_2,))
top_scorers_2 = cursor.fetchall()
top_scorers_df_2 = pd.DataFrame(top_scorers_2, columns=['Игрок', 'Голы'])

# Get top assists for team_id_2
cursor.execute(query_top_assists, (team_id_2,))
top_assists_2 = cursor.fetchall()
top_assists_df_2 = pd.DataFrame(top_assists_2, columns=['Игрок', 'Ассисты'])

# Get overall top scorers
cursor.execute(query_overall_top_scorers)
overall_scorers = cursor.fetchall()
overall_scorers_df = pd.DataFrame(overall_scorers, columns=['Игрок', 'Голы', 'Общий рейтинг'])

# Get overall top assists
cursor.execute(query_overall_top_assists)
overall_assists = cursor.fetchall()
overall_assists_df = pd.DataFrame(overall_assists, columns=['Игрок', 'Ассисты', 'Общий рейтинг'])

# Merge team-specific data with overall ranking
top_scorers_df_1 = top_scorers_df_1.merge(overall_scorers_df[['Игрок', 'Общий рейтинг']], on='Игрок', how='left')
top_assists_df_1 = top_assists_df_1.merge(overall_assists_df[['Игрок', 'Общий рейтинг']], on='Игрок', how='left')
top_scorers_df_2 = top_scorers_df_2.merge(overall_scorers_df[['Игрок', 'Общий рейтинг']], on='Игрок', how='left')
top_assists_df_2 = top_assists_df_2.merge(overall_assists_df[['Игрок', 'Общий рейтинг']], on='Игрок', how='left')

print(f"\nТоп-3 бомбардира для {team_name_1} (с общим рейтингом):")
print(top_scorers_df_1.to_string(index=False))
print(f"\nТоп-3 ассистента для {team_name_1} (с общим рейтингом):")
print(top_assists_df_1.to_string(index=False))
print(f"\nТоп-3 бомбардира для {team_name_2} (с общим рейтингом):")
print(top_scorers_df_2.to_string(index=False))
print(f"\nТоп-3 ассистента для {team_name_2} (с общим рейтингом):")
print(top_assists_df_2.to_string(index=False))

# Calculate and rank average match statistics
query_avg_stats = '''
WITH TeamAggregatedStats AS (
    SELECT
        MatchStats.team_id,
        COUNT(DISTINCT MatchStats.match_id) AS total_matches,
        SUM(goals) AS total_goals,
        SUM(xg) AS total_xg,
        SUM(fouls) AS total_fouls,
        SUM(yellow_cards) AS total_yellow_cards,
        SUM(red_cards) AS total_red_cards,
        SUM(corners) AS total_corners,
        SUM(shots) AS total_shots,
        SUM(passes) AS total_passes,
        SUM(dribbles) AS total_dribbles,
        SUM(ppda) AS total_ppda,
        SUM(deep) AS total_deep
    FROM MatchStats JOIN Matches ON Matches.match_id = MatchStats.match_id
    WHERE season = (SELECT max(season) FROM Matches)
    GROUP BY team_id
),
TeamAverageStats AS (
    SELECT
        ts.team_id,
        t.team_name,
        ts.total_matches,
        CAST(ts.total_goals AS REAL) / ts.total_matches AS avg_goals,
        CAST(ts.total_xg AS REAL) / ts.total_matches AS avg_xg,
        CAST(ts.total_fouls AS REAL) / ts.total_matches AS avg_fouls,
        CAST(ts.total_yellow_cards AS REAL) / ts.total_matches AS avg_yellow_cards,
        CAST(ts.total_red_cards AS REAL) / ts.total_matches AS avg_red_cards,
        CAST(ts.total_corners AS REAL) / ts.total_matches AS avg_corners,
        CAST(ts.total_shots AS REAL) / ts.total_matches AS avg_shots,
        CAST(ts.total_passes AS REAL) / ts.total_matches AS avg_passes,
        CAST(ts.total_dribbles AS REAL) / ts.total_matches AS avg_dribbles,
        CAST(ts.total_ppda AS REAL) / ts.total_matches AS avg_ppda,
        CAST(ts.total_deep AS REAL) / ts.total_matches AS avg_deep
    FROM TeamAggregatedStats AS ts
    JOIN Teams AS t ON ts.team_id = t.team_id
)
SELECT
    tas.team_name AS 'Название команды',
    tas.total_matches AS 'Сыграно матчей',
    ROUND(tas.avg_goals, 2) AS 'Средние голы',
    RANK() OVER (ORDER BY tas.avg_goals DESC) AS 'Рейтинг по голам',
    ROUND(tas.avg_xg, 2) AS 'Средний xG',
    RANK() OVER (ORDER BY tas.avg_xg DESC) AS 'Рейтинг по xG',
    ROUND(tas.avg_fouls, 2) AS 'Средние фолы',
    RANK() OVER (ORDER BY tas.avg_fouls ASC) AS 'Рейтинг по фолам',
    ROUND(tas.avg_yellow_cards, 2) AS 'Средние желтые карточки',
    RANK() OVER (ORDER BY tas.avg_yellow_cards ASC) AS 'Рейтинг по ЖК',
    ROUND(tas.avg_red_cards, 2) AS 'Средние красные карточки',
    RANK() OVER (ORDER BY tas.avg_red_cards ASC) AS 'Рейтинг по КК',
    ROUND(tas.avg_corners, 2) AS 'Средние угловые',
    RANK() OVER (ORDER BY tas.avg_corners DESC) AS 'Рейтинг по угловым',
    ROUND(tas.avg_shots, 2) AS 'Средние удары',
    RANK() OVER (ORDER BY tas.avg_shots DESC) AS 'Рейтинг по ударам',
    ROUND(tas.avg_passes, 2) AS 'Средние передачи',
    RANK() OVER (ORDER BY tas.avg_passes DESC) AS 'Рейтинг по передачам',
    ROUND(tas.avg_dribbles, 2) AS 'Средний дриблинг',
    RANK() OVER (ORDER BY tas.avg_dribbles DESC) AS 'Рейтинг по дриблингу',
    ROUND(tas.avg_ppda, 2) AS 'Средний PPDA',
    RANK() OVER (ORDER BY tas.avg_ppda ASC) AS 'Рейтинг по PPDA',
    ROUND(tas.avg_deep, 2) AS 'Средний deep',
    RANK() OVER (ORDER BY tas.avg_deep DESC) AS 'Рейтинг по deep',
    tas.team_id
FROM TeamAverageStats AS tas
ORDER BY tas.team_name ASC
'''

cursor.execute(query_avg_stats)
avg_stats_data_all = cursor.fetchall()

columns = [
    'Название команды', 'Сыграно матчей',
    'Средние голы', 'Рейтинг по голам',
    'Средний xG', 'Рейтинг по xG',
    'Средние фолы', 'Рейтинг по фолам',
    'Средние желтые карточки', 'Рейтинг по ЖК',
    'Средние красные карточки', 'Рейтинг по КК',
    'Средние угловые', 'Рейтинг по угловым',
    'Средние удары', 'Рейтинг по ударам',
    'Средние передачи', 'Рейтинг по передачам',
    'Средний дриблинг', 'Рейтинг по дриблингу',
    'Средний PPDA', 'Рейтинг по PPDA',
    'Средний deep', 'Рейтинг по deep',
    'team_id'
]
avg_stats_df_all = pd.DataFrame(avg_stats_data_all, columns=columns)

team1_avg_stats = avg_stats_df_all[avg_stats_df_all['team_id'] == team_id_1].iloc[0]
team2_avg_stats = avg_stats_df_all[avg_stats_df_all['team_id'] == team_id_2].iloc[0]

print("\nСредняя статистика за матч и их рейтинги:")

# This list should contain only the base stat labels, not ranks
base_stat_labels_map = {
    'Средние голы': 'Рейтинг по голам',
    'Средний xG': 'Рейтинг по xG',
    'Средние фолы': 'Рейтинг по фолам',
    'Средние желтые карточки': 'Рейтинг по ЖК',
    'Средние красные карточки': 'Рейтинг по КК',
    'Средние угловые': 'Рейтинг по угловым',
    'Средние удары': 'Рейтинг по ударам',
    'Средние передачи': 'Рейтинг по передачам',
    'Средний дриблинг': 'Рейтинг по дриблингу',
    'Средний PPDA': 'Рейтинг по PPDA',
    'Средний deep': 'Рейтинг по deep'
}

formatted_stats_for_display = []

for base_label, rank_label in base_stat_labels_map.items():
    val1_formatted = f"{team1_avg_stats[base_label]} ({team1_avg_stats[rank_label]})"
    val2_formatted = f"{team2_avg_stats[base_label]} ({team2_avg_stats[rank_label]})"

    formatted_stats_for_display.append({
        'label': base_label,
        'val1_display': val1_formatted,
        'val2_display': val2_formatted
    })

# Determine max width for labels and values for alignment using get_display_width
max_label_len_avg = max(get_display_width(stat['label']) for stat in formatted_stats_for_display)
max_val_len_avg = 0
for stat in formatted_stats_for_display:
    max_val_len_avg = max(max_val_len_avg, get_display_width(stat['val1_display']), get_display_width(stat['val2_display']))

# Print header for average stats
header_avg_home_text = team_name_1
header_avg_away_text = team_name_2
# Manually construct header line with proper padding for team names and middle empty space
team1_header_padded_avg = header_avg_home_text + ' ' * (max_val_len_avg - get_display_width(header_avg_home_text))
team2_header_padded_avg = ' ' * (max_val_len_avg - get_display_width(header_avg_away_text)) + header_avg_away_text
middle_empty_space_avg = ' ' * max_label_len_avg

print(f"{team1_header_padded_avg} | {middle_empty_space_avg} | {team2_header_padded_avg}")
print(f"{'-' * max_val_len_avg}-+-{'-' * max_label_len_avg}-+-{'-' * max_val_len_avg}")

for stat in formatted_stats_for_display:
    val1_display = stat['val1_display']
    label_display = stat['label']
    val2_display = stat['val2_display']

    val1_padded = val1_display + ' ' * (max_val_len_avg - get_display_width(val1_display))

    # For centering label_display, calculate padding
    label_padding_left = (max_label_len_avg - get_display_width(label_display)) // 2
    label_padding_right = max_label_len_avg - get_display_width(label_display) - label_padding_left
    label_padded = ' ' * label_padding_left + label_display + ' ' * label_padding_right

    val2_padded = val2_display + ' ' * (max_val_len_avg - get_display_width(val2_display))

    print(f"{val1_padded} | {label_padded} | {val2_padded}")

# Get last head-to-head match details
# Modified query to fetch up to 2 most recent matches
query_last_match = '''
SELECT
    match_id,
    match_date,
    team1_id AS home_team_id,
    team2_id AS away_team_id
FROM Matches
WHERE
    (team1_id = ? AND team2_id = ?)
    OR
    (team1_id = ? AND team2_id = ?)
ORDER BY match_date DESC
LIMIT 2;  -- Changed from LIMIT 1 to LIMIT 2
'''

cursor.execute(query_last_match, (team_id_1, team_id_2, team_id_2, team_id_1))
h2h_matches_candidates = cursor.fetchall() # Renamed to plural

found_detailed_match = False
selected_match_id = None
selected_match_date = None
selected_home_team_id = None
selected_away_team_id = None
squad_data = []
events_data = []
h2h_stats_data = []

for match_candidate in h2h_matches_candidates:
    current_match_id = match_candidate[0]
    current_match_date = match_candidate[1]
    current_home_team_id = match_candidate[2]
    current_away_team_id = match_candidate[3]

    print(f"\nПопытка получения данных для очного матча ID: {current_match_id} (дата: {current_match_date})")

    # --- Fetch Squad Data ---
    query_squad = '''
    SELECT
        PS.position_on_field,
        P.player_name,
        PS.minutes_played,
        PS.goals,
        PS.assists,
        PS.yellow_card,
        PS.red_card,
        PS.team_id
    FROM PlayerStats AS PS
    JOIN Players AS P ON PS.player_id = P.player_id
    WHERE PS.match_id = ?
    ORDER BY PS.player_stat_id ASC;
    '''
    cursor.execute(query_squad, (current_match_id,))
    current_squad_data = cursor.fetchall()

    # --- Fetch Events Data ---
    query_events = '''
    SELECT
        PA.action_minute,
        PA.action_type,
        P.player_name,
        PA.team_id,
        APP.player_name AS assistant_name
    FROM PlayerActions AS PA
    JOIN Players AS P ON PA.player_id = P.player_id
    LEFT JOIN Players AS APP ON PA.assist_by = APP.player_id
    WHERE PA.match_id = ?
    ORDER BY PA.action_minute ASC;
    '''
    cursor.execute(query_events, (current_match_id,))
    current_events_data = cursor.fetchall()

    # --- Fetch H2H Stats Data ---
    query_h2h_stats = '''
    SELECT
        team_id,
        goals, xg, possession, yellow_cards, corners, offsides,
        shots, shots_on_target, passes, accurate_passes,
        dribbles, dribbles_succ, ppda, deep
    FROM MatchStats
    WHERE match_id = ? AND team_id IN (?, ?)
    ORDER BY team_id ASC;
    '''
    cursor.execute(query_h2h_stats, (current_match_id, team_id_1, team_id_2))
    current_h2h_stats_data = cursor.fetchall()

    if current_squad_data and current_events_data and len(current_h2h_stats_data) == 2:
        print(f"Подробные данные найдены для матча ID: {current_match_id}.")
        selected_match_id = current_match_id
        selected_match_date = current_match_date
        selected_home_team_id = current_home_team_id
        selected_away_team_id = current_away_team_id
        squad_data = current_squad_data
        events_data = current_events_data
        h2h_stats_data = current_h2h_stats_data
        found_detailed_match = True
        break # Found a match with all data, stop searching
    else:
        print(f"Неполные данные для матча ID: {current_match_id}. Продолжение поиска...")

if not found_detailed_match:
    print("\nНе удалось найти очный матч с полными данными (состав, события, статистика) среди последних двух.\n")
elif selected_match_id is not None: # Ensure a match was actually found before proceeding
    # Use selected_match_id, selected_match_date, etc. for further processing
    # The rest of the original code that prints out squad, events, and stats will now
    # operate on these `selected_` variables and `squad_data`, `events_data`, `h2h_stats_data`.

    print(f"\nПоследний очный матч (ID: {selected_match_id}) состоялся {selected_match_date}")

    # Retrieve actual team names for the specific match
    cursor.execute("SELECT team_name FROM Teams WHERE team_id = ?", (selected_home_team_id,))
    actual_home_team_name = cursor.fetchone()[0]
    cursor.execute("SELECT team_name FROM Teams WHERE team_id = ?", (selected_away_team_id,))
    actual_away_team_name = cursor.fetchone()[0]

    print(f"Домашняя команда: {actual_home_team_name} (ID: {selected_home_team_id})")
    print(f"Гостевая команда: {actual_away_team_name} (ID: {selected_away_team_id})")

    # Squad data printing (moved into this else block)
    if squad_data: # now checking the `squad_data` collected from the loop
        squad_columns = ['Позиция', 'Имя Игрока', 'Минуты', 'Голы', 'Ассисты', 'ЖК', 'КК', 'ID Команды']

        home_squad_data = [row for row in squad_data if row[7] == selected_home_team_id] # Use selected_home_team_id
        away_squad_data = [row for row in squad_data if row[7] == selected_away_team_id] # Use selected_away_team_id

        home_squad_df = pd.DataFrame(home_squad_data, columns=squad_columns).drop(columns=['ID Команды'])
        away_squad_df = pd.DataFrame(away_squad_data, columns=squad_columns).drop(columns=['ID Команды'])

        def format_goals_assists(row):
            goals_str = '⚽' * row['Голы'] if row['Голы'] > 0 else ''
            assists_str = '👟' * row['Ассисты'] if row['Ассисты'] > 0 else ''
            return goals_str + assists_str

        def format_cards(row):
            yellow_cards_str = '🟨' * row['ЖК'] if row['ЖК'] > 0 else ''
            red_cards_str = '🟥' * row['КК'] if row['КК'] > 0 else ''
            return yellow_cards_str + red_cards_str

        home_squad_df['Голы/Ассисты'] = home_squad_df.apply(format_goals_assists, axis=1)
        home_squad_df['Карточки'] = home_squad_df.apply(format_cards, axis=1)
        away_squad_df['Голы/Ассисты'] = away_squad_df.apply(format_goals_assists, axis=1)
        away_squad_df['Карточки'] = away_squad_df.apply(format_cards, axis=1)

        home_squad_df = home_squad_df.drop(columns=['Голы', 'Ассисты', 'ЖК', 'КК'])
        away_squad_df = away_squad_df.drop(columns=['Голы', 'Ассисты', 'ЖК', 'КК'])

        print(f"\nСостав команд в матче {selected_match_id} (отсортировано по ID статистики игрока):")

        combined_headers = ['Позиция', 'Имя Игрока', 'Минуты', 'Голы/Ассисты', 'Карточки']

        max_widths = {col: len(col) for col in combined_headers}
        for df_temp in [home_squad_df, away_squad_df]:
            for col in combined_headers:
                if col in df_temp.columns and not df_temp.empty:
                    max_widths[col] = max(max_widths[col], df_temp[col].astype(str).apply(get_display_width).max())

        home_header_line_parts = []
        away_header_line_parts = []
        for col in combined_headers:
            home_header_line_parts.append(col + ' ' * (max_widths[col] - get_display_width(col)))
            away_header_line_parts.append(col + ' ' * (max_widths[col] - get_display_width(col)))
        home_header_line = "  ".join(home_header_line_parts)
        away_header_line = "  ".join(away_header_line_parts)

        total_home_width = get_display_width(home_header_line)
        total_away_width = get_display_width(away_header_line)

        header_home_text = f"{actual_home_team_name} (Домашняя команда)"
        header_away_text = f"{actual_away_team_name} (Гостевая команда)"

        print(f"\n{header_home_text + ' ' * (total_home_width - get_display_width(header_home_text))} | {header_away_text + ' ' * (total_away_width - get_display_width(header_away_text))}")
        print(f"{'=' * total_home_width} + {'=' * total_away_width}")
        print(home_header_line + ' ' * (total_home_width - get_display_width(home_header_line)) + ' | ' + away_header_line + ' ' * (total_away_width - get_display_width(away_header_line)))
        print(f"{'=' * total_home_width} + {'=' * total_away_width}")

        max_rows = max(len(home_squad_df), len(away_squad_df))
        for i in range(max_rows):
            home_row_str_parts = []
            if i < len(home_squad_df):
                for col in combined_headers:
                    val = str(home_squad_df.iloc[i][col])
                    home_row_str_parts.append(val + ' ' * (max_widths[col] - get_display_width(val)))
            else:
                home_row_str_parts = [' ' * max_widths[col] for col in combined_headers]
            home_row_str = "  ".join(home_row_str_parts)

            away_row_str_parts = []
            if i < len(away_squad_df):
                for col in combined_headers:
                    val = str(away_squad_df.iloc[i][col])
                    away_row_str_parts.append(val + ' ' * (max_widths[col] - get_display_width(val)))
            else:
                away_row_str_parts = [' ' * max_widths[col] for col in combined_headers]
            away_row_str = "  ".join(away_row_str_parts)

            print(home_row_str + ' ' * (total_home_width - get_display_width(home_row_str)) + ' | ' + away_row_str + ' ' * (total_away_width - get_display_width(away_row_str)))

    else:
        print(f"Данные по составу для матча {selected_match_id} не найдены.")

    # Events data printing (moved into this else block)
    if events_data: # now checking the `events_data` collected from the loop
        print(f"\nСобытия матча {selected_match_id} (хронологический порядок):")

        formatted_events_with_team = []
        max_home_event_len = 0

        for minute, action_type, player_name, team_id, assistant_name in events_data:
            if action_type == 'Assist':
                continue

            event_text = f"Минута {minute}: "
            if action_type == 'Goal':
                event_text += f"⚽ Гол - {player_name}"
                if assistant_name:
                    event_text += f" (ассист: {assistant_name})"
            elif action_type == 'Yellow card':
                event_text += f"🟨 Желтая карточка - {player_name}"
            elif action_type == 'Red card':
                event_text += f"🟥 Красная карточка - {player_name}"
            else:
                event_text += f"{action_type} - {player_name}"

            formatted_events_with_team.append((team_id, event_text))

            if team_id == selected_home_team_id and get_display_width(event_text) > max_home_event_len:
                max_home_event_len = get_display_width(event_text)

        if max_home_event_len < get_display_width(f"{actual_home_team_name} (Домашняя команда)"):
            max_home_event_len = get_display_width(f"{actual_home_team_name} (Домашняя команда)")

        header_home = f"{actual_home_team_name} (Домашняя команда)"
        header_away = f"{actual_away_team_name} (Гостевая команда)"
        print(f"{header_home + ' ' * (max_home_event_len - get_display_width(header_home))} | {header_away}")
        print(f"-{'-' * max_home_event_len}-+-{'-' * get_display_width(header_away)}")

        for team_id, event_text in formatted_events_with_team:
            if team_id == selected_home_team_id:
                print(event_text + ' ' * (max_home_event_len - get_display_width(event_text)) + ' |')
            elif team_id == selected_away_team_id:
                print(f"{'' :<{max_home_event_len}} | {event_text}")
    else:
        print(f"События для матча {selected_match_id} не найдены.")

    # H2H stats printing (moved into this else block)
    if h2h_stats_data and len(h2h_stats_data) == 2: # now checking `h2h_stats_data` collected from the loop
        home_stats_row = h2h_stats_data[0] if h2h_stats_data[0][0] == selected_home_team_id else h2h_stats_data[1]
        away_stats_row = h2h_stats_data[1] if h2h_stats_data[1][0] == selected_away_team_id else h2h_stats_data[0]

        print(f"\nСтатистика последнего очного матча (ID: {selected_match_id}):")

        stats_labels = [
            "Голы", "xG", "Владение мячом", "Желтые карточки", "Угловые", "Оффсайды",
            "Удары", "Передачи", "Дриблинг", "PPDA", "Deep"
        ]

        home_stats_initial = list(home_stats_row[1:])
        away_stats_initial = list(away_stats_row[1:])

        home_stats = [
            home_stats_initial[0], home_stats_initial[1], home_stats_initial[2], home_stats_initial[3],
            home_stats_initial[4], home_stats_initial[5],
            f"{home_stats_initial[6]} ({home_stats_initial[7]} точных)",
            f"{home_stats_initial[8]} ({home_stats_initial[9]} точных)",
            f"{home_stats_initial[10]} ({home_stats_initial[11]} успешных)",
            home_stats_initial[12], home_stats_initial[13]
        ]

        away_stats = [
            away_stats_initial[0], away_stats_initial[1], away_stats_initial[2], away_stats_initial[3],
            away_stats_initial[4], away_stats_initial[5],
            f"{away_stats_initial[6]} ({away_stats_initial[7]} точных)",
            f"{away_stats_initial[8]} ({away_stats_initial[9]} точных)",
            f"{away_stats_initial[10]} ({away_stats_initial[11]} успешных)",
            away_stats_initial[12], away_stats_initial[13]
        ]

        max_label_len = max(len(label) for label in stats_labels)
        max_val_len = 0
        for i in range(len(stats_labels)):
            max_val_len = max(max_val_len, get_display_width(str(home_stats[i])), get_display_width(str(away_stats[i])))

        print(f"{actual_home_team_name + ' ' * (max_val_len - get_display_width(actual_home_team_name))} | {'':^{max_label_len}} | {actual_away_team_name + ' ' * (max_val_len - get_display_width(actual_away_team_name))}")
        print(f"{'-' * max_val_len}-+-{'-' * max_label_len}-+-{'-' * max_val_len}")

        for i, label in enumerate(stats_labels):
            val1 = str(home_stats[i])
            val2 = str(away_stats[i])
            print(val1 + ' ' * (max_val_len - get_display_width(val1)) + f" | {label:^{max_label_len}} | " + val2 + ' ' * (max_val_len - get_display_width(val2)))
    elif h2h_stats_data:
        print("\nДанные статистики найдены только для одной из команд в матче {selected_match_id}.")
    else:
        print("\nСтатистика для матча {selected_match_id} не найдена.")

conn.close()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Успешно подключено к /content/drive/MyDrive/Data_Bases/rpl.db
Successfully connected to FA_Premier_League.db and created a cursor object.
Введите ID первой команды для сравнения: 1
Введите ID второй команды для сравнения: 3
Вы выбрали команды с ID: 1 и 3
Команда 1: CSKA (ID: 1)
Команда 2: Krasnodar (ID: 3)

Турнирная таблица:
 position team_name  played  won  drawn  lost  points goals
        4      CSKA      13    7      5     1      26 21-12
        5 Krasnodar      13    6      5     2      23 21-16

Последние 5 матчей для CSKA:
      Дата   Домашняя команда  Голы домашней команды Гостевая команда  Голы гостевой команды Результат для команды
2026-11-01           Orenburg                      0             CSKA                      0                 Ничья
2026-10-25               CSKA                      2          Spartak                      0         